# Python Dunder Methods — Interview Preparation

Covers: __repr__/__str__, container protocol, iterator protocol, context manager, comparison, arithmetic, __call__, __hash__, and interview Q&A.

## 1. __repr__ and __str__

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        """Unambiguous — for developers, used in REPL and containers"""
        return f'Vector({self.x!r}, {self.y!r})'

    def __str__(self):
        """Human-readable — used in print() and str()"""
        return f'<{self.x}, {self.y}>'


v = Vector(3, 4)
print('repr(v):', repr(v))   # Vector(3, 4)
print('str(v):', str(v))     # <3, 4>
print('print(v):', v)        # <3, 4>  — uses __str__
print('[v]:', [v])           # [Vector(3, 4)]  — uses __repr__ in containers
print(f'f-string: {v}')      # <3, 4>  — uses __str__

## 2. Container Protocol

In [ ]:
class NumberList:
    """Custom container implementing the full container protocol"""
    def __init__(self, *args):
        self._data = list(args)

    def __len__(self):
        return len(self._data)

    def __getitem__(self, index):
        return self._data[index]  # supports slicing too

    def __setitem__(self, index, value):
        self._data[index] = value

    def __delitem__(self, index):
        del self._data[index]

    def __contains__(self, item):
        return item in self._data

    def __iter__(self):
        return iter(self._data)

    def __reversed__(self):
        return reversed(self._data)

    def __repr__(self):
        return f'NumberList({self._data!r})'


nl = NumberList(1, 2, 3, 4, 5)
print('len:', len(nl))           # __len__
print('nl[0]:', nl[0])           # __getitem__
print('nl[1:3]:', nl[1:3])       # __getitem__ with slice
print('3 in nl:', 3 in nl)       # __contains__
print('list(nl):', list(nl))     # __iter__
print('reversed:', list(reversed(nl)))  # __reversed__

nl[0] = 99                       # __setitem__
print('After nl[0]=99:', nl)

del nl[0]                        # __delitem__
print('After del nl[0]:', nl)

## 3. Iterator Protocol

In [ ]:
class Fibonacci:
    """Iterator that generates Fibonacci numbers up to a limit"""
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1

    def __iter__(self):
        return self  # iterator returns itself

    def __next__(self):
        if self.a > self.limit:
            raise StopIteration
        value = self.a
        self.a, self.b = self.b, self.a + self.b
        return value


fib = Fibonacci(100)
print('Fibonacci up to 100:', list(fib))

# Manual iteration
fib2 = Fibonacci(10)
print('Manual next():', next(fib2), next(fib2), next(fib2))

# next() with default — no StopIteration
it = iter([1, 2])
print(next(it, 'done'), next(it, 'done'), next(it, 'done'))

## 4. Context Manager Protocol

In [ ]:
import time

class Timer:
    """Context manager for timing code blocks"""
    def __enter__(self):
        self.start = time.perf_counter()
        return self  # bound to 'as' variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f'Elapsed: {self.elapsed:.6f}s')
        return False  # don't suppress exceptions


with Timer() as t:
    total = sum(range(1_000_000))

print(f'Computed sum: {total}')
print(f'Elapsed (from t): {t.elapsed:.6f}s')

In [ ]:
class SuppressErrors:
    """Context manager that suppresses specified exceptions"""
    def __init__(self, *exceptions):
        self.exceptions = exceptions

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type and issubclass(exc_type, self.exceptions):
            print(f'Suppressed {exc_type.__name__}: {exc_val}')
            return True  # suppress the exception
        return False  # propagate other exceptions


with SuppressErrors(ValueError, ZeroDivisionError):
    x = 1 / 0  # ZeroDivisionError — suppressed!

print('Code continues after suppressed exception')

## 5. Comparison and Arithmetic Operators

In [ ]:
from functools import total_ordering

@total_ordering
class Money:
    def __init__(self, amount, currency='USD'):
        self.amount = amount
        self.currency = currency

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount == other.amount and self.currency == other.currency

    def __lt__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        if self.currency != other.currency:
            raise ValueError('Cannot compare different currencies')
        return self.amount < other.amount

    def __add__(self, other):
        if isinstance(other, Money):
            if self.currency != other.currency:
                raise ValueError('Cannot add different currencies')
            return Money(self.amount + other.amount, self.currency)
        return NotImplemented

    def __mul__(self, scalar):
        if isinstance(scalar, (int, float)):
            return Money(self.amount * scalar, self.currency)
        return NotImplemented

    def __rmul__(self, scalar):
        return self.__mul__(scalar)

    def __hash__(self):
        return hash((self.amount, self.currency))

    def __repr__(self):
        return f'Money({self.amount}, {self.currency!r})'


m1 = Money(10, 'USD')
m2 = Money(20, 'USD')
m3 = Money(10, 'USD')

print('m1 == m3:', m1 == m3)   # True
print('m1 < m2:', m1 < m2)    # True
print('m1 > m2:', m1 > m2)    # False — generated by @total_ordering
print('m1 + m2:', m1 + m2)    # Money(30, 'USD')
print('m1 * 3:', m1 * 3)      # Money(30, 'USD')
print('3 * m1:', 3 * m1)      # Money(30, 'USD') — uses __rmul__
print('sorted:', sorted([m2, m1, m3]))

## 6. __call__ and __hash__

In [ ]:
# __call__ — makes instance callable
class Memoize:
    """Callable class that caches results"""
    def __init__(self, func):
        self.func = func
        self.cache = {}
        self.__doc__ = func.__doc__

    def __call__(self, *args):
        if args not in self.cache:
            self.cache[args] = self.func(*args)
        return self.cache[args]


@Memoize
def fib(n):
    """Fibonacci with memoization"""
    return n if n < 2 else fib(n-1) + fib(n-2)


print('fib(10):', fib(10))
print('fib(20):', fib(20))
print('callable(fib):', callable(fib))
print('cache size:', len(fib.cache))

In [ ]:
# __hash__ — required when defining __eq__
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        return isinstance(other, Point) and self.x == other.x and self.y == other.y

    def __hash__(self):
        # Use same fields as __eq__
        return hash((self.x, self.y))

    def __repr__(self):
        return f'Point({self.x}, {self.y})'


p1 = Point(1, 2)
p2 = Point(1, 2)
p3 = Point(3, 4)

print('p1 == p2:', p1 == p2)              # True
print('hash(p1) == hash(p2):', hash(p1) == hash(p2))  # True
print('set:', {p1, p2, p3})              # {Point(1, 2), Point(3, 4)}
d = {p1: 'origin'}
print('d[p2]:', d[p2])                   # 'origin' — p2 finds p1's entry